<h3>문서와 메타데이터 준비</h3>
<p><strong>개념</strong> 문서 본문과 출처 정보는 함께 이동해야 합니다. 먼저 실행 위치와 실제 문서 경로를 확인한 뒤 파일명, 확장자, 쪽 번호를 메타데이터로 정리합니다.</p>

In [1]:
# sys는 현재 노트북 커널의 Python 정보를 제공합니다.
# version.split()[0]은 긴 버전 문자열에서 버전 번호만 고릅니다.
import sys
print("Python:", sys.version.split()[0])


Python: 3.12.13


<h3>노트북 실행 위치 확인</h3>
<p>이 노트북은 저장소 안 어느 폴더에서 시작해도 됩니다. <code>Path.cwd()</code>는 현재 실행 폴더를 나타내며, 아래 셀은 상위 폴더를 차례로 확인해 제공된 PDF를 찾습니다.</p>

In [2]:
# Path는 파일과 폴더 경로를 운영체제에 맞게 다루는 객체입니다.
# parents는 현재 폴더의 상위 폴더들을 가까운 순서로 제공합니다.
from pathlib import Path

repo = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    data_dir = candidate / "labs/data/rag-corpus"
    if data_dir.is_dir():
        repo = candidate
        break
if repo is None:
    raise FileNotFoundError("저장소 안에서 노트북을 실행하세요.")
source = repo / "labs/data/rag-corpus/pdf/easy/E02.pdf"
if not source.is_file():
    raise FileNotFoundError(source)
print("현재 폴더 이름:", Path.cwd().name)
print("실습 문서:", source.relative_to(repo))


현재 폴더 이름: notebooks
실습 문서: labs/data/rag-corpus/pdf/easy/E02.pdf


<h3>경로 객체의 값 관찰</h3>
<p><code>name</code>은 파일명 전체, <code>stem</code>은 확장자를 뺀 이름, <code>suffix</code>는 점을 포함한 확장자입니다. 속성은 객체가 가진 값을 읽는 이름입니다.</p>

In [3]:
# source는 실제 파일을 가리키는 Path 객체입니다.
# type()은 객체의 자료형을, 각 속성은 경로에서 분리한 문자열을 반환합니다.
print("객체:", type(source).__name__)
print("name:", source.name)
print("stem:", source.stem)
print("suffix:", source.suffix)


객체: PosixPath
name: E02.pdf
stem: E02
suffix: .pdf


<h3>공통 메타데이터 만들기</h3>
<p>딕셔너리는 이름표인 키와 값의 쌍을 저장합니다. 로더가 문자열로 준 쪽 번호와 태그 목록을 검색 단계에서 쓰기 쉬운 형식으로 바꿉니다.</p>

In [4]:
# raw는 로더가 반환한 원래 값을 나타내는 딕셔너리입니다.
# int()는 쪽 번호를 정수로, join()은 태그 목록을 쉼표로 연결한 문자열로 바꿉니다.
raw = {"page": "2", "tags": ["배송", "환불"]}
metadata = {
    "doc_id": source.stem,
    "source": source.name,
    "file_type": source.suffix.lstrip(".").lower(),
    "parser": "notebook-demo",
    "page": int(raw["page"]),
    "tags": ",".join(raw["tags"]),
}
print(metadata)


{'doc_id': 'E02', 'source': 'E02.pdf', 'file_type': 'pdf', 'parser': 'notebook-demo', 'page': 2, 'tags': '배송,환불'}


<h3>값 하나를 바꾸어 비교</h3>
<p>같은 문서에서 선택한 쪽만 2쪽에서 3쪽으로 바꿉니다. <code>{**metadata, "page": 3}</code>은 원본 딕셔너리를 수정하지 않고 새 딕셔너리를 만듭니다.</p>

In [5]:
# **metadata는 기존 키와 값을 새 딕셔너리에 펼칩니다.
# 뒤에 적은 page 값이 기존 page를 덮어쓰므로 두 결과를 비교할 수 있습니다.
page_2 = metadata
page_3 = {**metadata, "page": 3}
print("기본 선택:", page_2["page"])
print("변경 선택:", page_3["page"])
print("원본 유지:", metadata["page"])


기본 선택: 2
변경 선택: 3
원본 유지: 2


<h3>일부러 실패해 보기</h3>
<p><strong>Before 원인</strong> 쪽 번호를 0으로 저장하면 사람이 보는 문서 위치와 맞지 않습니다. 다음 셀은 허용 범위를 검사해 구체적인 오류를 포착합니다.</p>

In [6]:
# 잘못된 쪽 번호 0을 넣어 검증 실패를 재현합니다.
# try/except는 ValueError를 출력하고 다음 셀을 계속 실행하게 합니다.
try:
    page = 0
    if page < 1:
        raise ValueError("page는 1 이상이어야 합니다")
except ValueError as error:
    print(f"caught: {error}")


caught: page는 1 이상이어야 합니다


<p><strong>After 수정</strong> 같은 쪽 번호에 1 이상의 값을 사용하고, 메타데이터를 만든 뒤에도 값이 유효한지 확인합니다.</p>

In [7]:
# page를 1로 수정한 뒤 조건을 먼저 검증합니다.
# 새 딕셔너리의 page가 실제로 1인지 assert로 다시 확인합니다.
page = 1
assert page >= 1
fixed_metadata = {**metadata, "page": page}
assert fixed_metadata["page"] == 1
print("validated page:", fixed_metadata["page"])


validated page: 1
